# Cervical Cancer - cleaning

Follows `01_preprocessing_python.ipynb`: unusable columns, then duplicates, then
remaining missing values - in that order


## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

CLEAN_DIR = Path("../data/clean")
df = pd.read_csv(CLEAN_DIR / "cervical_cancer_clean.csv")
print("starting shape:", df.shape)


starting shape: (858, 36)


## 2. Columns to drop

`STDs:cervical condylomatosis`/`STDs:AIDS`: constant at 0, zero variance.
`STDs: Time since first/last diagnosis`: 91.7% missing, only 71 usable values.


In [2]:
cols_to_drop = [
    "STDs:cervical condylomatosis", "STDs:AIDS",
    "STDs: Time since first diagnosis", "STDs: Time since last diagnosis",
]
df = df.drop(columns=cols_to_drop)
print("after dropping columns:", df.shape)


after dropping columns: (858, 32)


## 3. Duplicates - checked before imputation

No patient ID, so a "duplicate" is an identical row, not a confirmed identical patient.


In [3]:
targets = ["Hinselmann", "Schiller", "Citology", "Biopsy"]
dup_mask = df.duplicated(keep=False)
print("rows in a duplicate group:", dup_mask.sum())
print(df.loc[dup_mask, targets].sum())


rows in a duplicate group: 43
Hinselmann    0
Schiller      2
Citology      2
Biopsy        2
dtype: int64


Dropped regardless of same-patient-or-coincidence: an identical row split across
train/test leaks into the evaluation either way.


In [4]:
before = len(df)
positives_before = df[targets].sum()
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print(f"{before} -> {len(df)} rows ({before - len(df)} dropped)")
print(df[targets].sum() - positives_before)


858 -> 835 rows (23 dropped)
Hinselmann    0
Schiller     -1
Citology     -1
Biopsy       -1
dtype: int64


Run before the fill in S4, not after: filling `NaN` the same way on two rows that
originally differed can make them look identical for no real reason. Checked both
orders - post-fill dedup would have found 70 "duplicates" instead of 43, 27 of them
manufactured by the fill choice alone.


## 4. Remaining missing values

`Smokes*`/`Hormonal Contraceptives*`/`IUD*`/`STDs*`: missing = not applicable, filled
with 0. `Number of sexual partners`/`First sexual intercourse`/`Num of pregnancies`:
missing = genuinely unknown, filled with median.


In [5]:
cols_fill_zero = [
    "Smokes", "Smokes (years)", "Smokes (packs/year)",
    "Hormonal Contraceptives", "Hormonal Contraceptives (years)",
    "IUD", "IUD (years)",
    "STDs", "STDs (number)", "STDs:condylomatosis",
    "STDs:vaginal condylomatosis", "STDs:vulvo-perineal condylomatosis",
    "STDs:syphilis", "STDs:pelvic inflammatory disease",
    "STDs:genital herpes", "STDs:molluscum contagiosum",
    "STDs:HIV", "STDs:Hepatitis B", "STDs:HPV",
]
df[cols_fill_zero] = df[cols_fill_zero].fillna(0)

cols_fill_median = ["Number of sexual partners", "First sexual intercourse", "Num of pregnancies"]
for c in cols_fill_median:
    df[c] = df[c].fillna(df[c].median())

print("remaining missing values:", int(df.isna().sum().sum()))


remaining missing values: 0


Not re-running dedup after this fill - it would catch rows only made identical by our
own fill choice, not by the patients' actual answers.


## 5. Final check

In [6]:
print(f"Shape: {df.shape}")
print(f"Missing: {int(df.isna().sum().sum())}")


Shape: (835, 32)
Missing: 0


## 6. Export

`cervical_cancer_final.csv`/`.parquet` - use this one for feature selection.


In [7]:
df.to_csv(CLEAN_DIR / "cervical_cancer_final.csv", index=False)
df.to_parquet(CLEAN_DIR / "cervical_cancer_final.parquet", index=False)
print("exported:", df.shape)


exported: (835, 32)
